# State Revenue: Where the Money Comes From
General Fund plus special funds. The first chart includes fees and other revenue; the second isolates taxes.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
rev = pd.read_csv(PROCESSED / "state_revenue.csv")
rev = rev[rev.group != "Transfers"]
BREAK = {2016.5: "series break (LAO categories change)"}
allrev = add_views(rev.groupby(["fy", "tax_type"], as_index=False).amount_k.sum())
fig = stacked_area_toggle(allrev, "tax_type", "State revenue by source, all funds", breaks=BREAK)
fig.show()

In [2]:
taxes = add_views(rev[rev.group == "Tax"].groupby(["fy", "tax_type"], as_index=False).amount_k.sum())
fig = stacked_area_toggle(taxes, "tax_type", "State taxes by type, all funds",
                          "Taxes only (excludes fees, fines, investment income, transfers)", breaks=BREAK)
fig.show()

## How dependent is the state on the income tax?
Share of all state tax revenue (general + special funds) from each of the big three taxes.

In [3]:
tx = rev[rev.group == "Tax"].groupby(["fy", "tax_type"]).amount_k.sum().unstack().fillna(0)
share = tx.div(tx.sum(axis=1), axis=0) * 100
fig = go.Figure()
for i, c in enumerate(["Personal income tax", "Sales & use tax", "Corporation tax"]):
    fig.add_trace(go.Scatter(x=share.index, y=share[c], name=c, mode="lines", line=dict(color=PALETTE[i], width=2.5)))
fig.update_layout(yaxis_title="% of state tax revenue", xaxis_title="Fiscal year (ending June 30)")
fig.add_vline(x=2016.5, line_dash="dot", line_color="grey")
base_layout(fig, "Share of state tax revenue from the big three taxes", SOURCE_NOTE)
fig.show()

## Growth of each tax in real, per-person terms
Inflation-adjusted dollars per California resident, so growth reflects policy, rates and the economy rather than inflation or headcount.

In [4]:
pc = add_views(rev[rev.group == "Tax"].groupby(["fy", "tax_type"], as_index=False).amount_k.sum())
w = pc.pivot_table(index="fy", columns="tax_type", values="real_pc")
fig = go.Figure()
for i, c in enumerate(w.sum().sort_values(ascending=False).index[:8]):
    fig.add_trace(go.Scatter(x=w.index, y=w[c], name=c, mode="lines", line=dict(color=PALETTE[i], width=2)))
fig.update_layout(yaxis_title=f"{base_fy()} $ per person", xaxis_title="Fiscal year (ending June 30)", yaxis_tickformat="$,.0f")
fig.add_vline(x=2016.5, line_dash="dot", line_color="grey")
base_layout(fig, f"Real tax revenue per person, by tax ({base_fy()} dollars)", SOURCE_NOTE)
fig.show()